In [ ]:
# -*- coding: utf-8 -*-
"""
Full Fine-Tuning Baseline - CIFAR-100

Train ResNet-50 from ImageNet weights to measure accuracy, time, and energy.
"""

import os
import gc
import math
import random
import time
import copy

from tqdm import tqdm
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torch.optim.lr_scheduler import OneCycleLR

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models


# ================================================================
# GPU POWER MONITORING
# ================================================================
try:
    import pynvml

    pynvml.nvmlInit()
    _nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    NVML_AVAILABLE = True
    print("GPU power monitoring enabled")
except Exception:
    NVML_AVAILABLE = False
    _nvml_handle = None
    print("GPU power monitoring unavailable")


def read_gpu_power_w():
    if not NVML_AVAILABLE:
        return 0.0
    try:
        return pynvml.nvmlDeviceGetPowerUsage(_nvml_handle) / 1000.0
    except Exception:
        return 0.0


# ================================================================
# CONFIGURATION
# ================================================================
EXPERIMENT_NAME = "cifar100_resnet50_fullft"
SEED = 42

# Model
MODEL_DEPTH = 50
USE_PRETRAINED = True

# Training
BATCH_SIZE = 64
NUM_WORKERS = 2
EPOCHS = 10
MAX_LR = 3e-4
WEIGHT_DECAY = 1e-2
CLIP_NORM = 1.0

# Data
NUM_CLASSES = 100
DATA_DIR = "./data"
RESULT_CSV = "cifar100_resnet50_fullft_results.csv"
CHECKPOINT_PATH = "checkpoints/fullft_resnet50_cifar100.pth"

print(f"\n{'=' * 70}")
print("  FULL FINE-TUNING BASELINE - CIFAR-100")
print(f"{'=' * 70}")
print(f"Model: ResNet-{MODEL_DEPTH} (ImageNet pretrained)")
print(f"Training: {EPOCHS} epochs, all parameters trainable")
print(f"Classes: {NUM_CLASSES}")
print(f"NUM_WORKERS: {NUM_WORKERS}")
print("Purpose: Measure time and energy for comparison")
print(f"{'=' * 70}\n")


# ================================================================
# REPRODUCIBILITY
# ================================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")


# ================================================================
# DATA LOADING
# ================================================================
print("Loading CIFAR-100...")

mean = (0.5071, 0.4867, 0.4408)
std = (0.2675, 0.2565, 0.2761)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_dataset_full = torchvision.datasets.CIFAR100(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=transform_train,
)
test_dataset = torchvision.datasets.CIFAR100(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=transform_test,
)

train_size = int(0.80 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size
train_dataset, val_dataset = random_split(
    train_dataset_full,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

print(f"Train: {train_size}, Val: {val_size}, Test: {len(test_dataset)}\n")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


# ================================================================
# MODEL
# ================================================================
def get_resnet_cifar100(depth=50, pretrained=True):
    model_fns = {
        18: models.resnet18,
        50: models.resnet50,
        101: models.resnet101,
    }
    model = model_fns[depth](pretrained=pretrained)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable, total - trainable


model = get_resnet_cifar100(MODEL_DEPTH, USE_PRETRAINED).to(device)

for param in model.parameters():
    param.requires_grad = True

total_params, trainable_params, frozen_params = count_params(model)
print("[Model Summary]")
print(f"  Total params: {total_params:,} ({total_params / 1e6:.2f}M)")
print(f"  Trainable params: {trainable_params:,} (100.0%)")
print("  All parameters trainable for full fine-tuning\n")


# ================================================================
# TRAINING FUNCTIONS
# ================================================================
criterion = nn.CrossEntropyLoss()


def train_epoch(model, loader, optimizer, scheduler, device, power_log=None):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for inputs, targets in tqdm(loader, desc="Train", leave=False):
        if power_log is not None:
            now = time.time()
            power_log["energy_J"] += read_gpu_power_w() * (now - power_log["last_t"])
            power_log["last_t"] = now

        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)

        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()

        if CLIP_NORM:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)

        optimizer.step()
        if scheduler:
            scheduler.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    return total_loss / len(loader), 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for inputs, targets in tqdm(loader, desc="Eval", leave=False):
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    return total_loss / len(loader), 100.0 * correct / total


# ================================================================
# TRAINING LOOP
# ================================================================
print("=" * 70)
print("  TRAINING FULL FINE-TUNING BASELINE")
print("=" * 70 + "\n")

optimizer = optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=WEIGHT_DECAY)
scheduler = OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS,
    pct_start=0.15,
    div_factor=10,
    final_div_factor=100,
)

print(f"Training for {EPOCHS} epochs...\n")

start_time = time.time()
power_log = {"energy_J": 0.0, "last_t": start_time}

best_val_acc = 0.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        device,
        power_log=power_log,
    )
    val_loss, val_acc = evaluate(model, val_loader, device)

    print(
        f"Epoch {epoch}/{EPOCHS}: "
        f"train_acc={train_acc:.2f}% val_acc={val_acc:.2f}%"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())

training_time = time.time() - start_time
training_energy = power_log["energy_J"]

print("\nTraining complete")
print(f"  Best val accuracy: {best_val_acc:.2f}%")
print(f"  Training time: {training_time:.1f}s ({training_time / 60:.2f} min)")
print(f"  Training energy: {training_energy / 1000:.2f} kJ\n")

if best_state is not None:
    model.load_state_dict(best_state)

_, test_acc = evaluate(model, test_loader, device)

print("=" * 70)
print("  FINAL RESULTS")
print("=" * 70)
print(f"Val Accuracy: {best_val_acc:.2f}%")
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Training Time: {training_time:.1f}s ({training_time / 60:.2f} min)")
print(f"Training Energy: {training_energy / 1000:.2f} kJ")
print(f"Total Params: {total_params:,}")
print("=" * 70 + "\n")


# ================================================================
# SAVE CHECKPOINT
# ================================================================
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)
checkpoint = {
    "model_state": best_state if best_state is not None else model.state_dict(),
    "best_val_acc": best_val_acc,
    "fullft_test_acc": test_acc,
    "train_time_s": training_time,
    "energy_J": training_energy,
    "total_params": total_params,
}
torch.save(checkpoint, CHECKPOINT_PATH)
print(f"Checkpoint saved to {CHECKPOINT_PATH}\n")


# ================================================================
# SAVE RESULTS
# ================================================================
results = {
    "method": "Full_FT",
    "dataset": "cifar100",
    "val_acc": best_val_acc,
    "test_acc": test_acc,
    "training_time_s": training_time,
    "training_time_min": training_time / 60,
    "energy_J": training_energy,
    "energy_kJ": training_energy / 1000,
    "total_params": total_params,
    "trainable_params": trainable_params,
    "epochs": EPOCHS,
}

df = pd.DataFrame([results])
if os.path.exists(RESULT_CSV):
    existing_df = pd.read_csv(RESULT_CSV)
    df = pd.concat([existing_df, df], ignore_index=True)

df.to_csv(RESULT_CSV, index=False)
print(f"Results saved to {RESULT_CSV}\n")

print("Full Fine-Tuning Baseline Complete")
print("Use these metrics for comparison with LoRA and CDWF")